# Study 0 corrected rerun, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

This run is **text only**. No compressor. No KV relay.

Uses only `data/plan_dev.jsonl` (12 families, 36 examples: base, A-flip, B-flip). Do **not** load `data/test.jsonl`, `data/hop_test.jsonl`, or a `plan_test` file.

Writes `/kaggle/working/results_stage_plan0_corrected`. Do **not** resume `results_stage_plan0`; keep that first run as-is.

Fixes relative to the first run: explicit lowest-cost objective; same receiver wrapper for collaboration and oracle, with eligibility as a mandatory constraint; extra B compute is a planning pass then an answer pass; compact A messages with a 256-token allowance.

Primary score remains cheapest eligible carrier within budget. `feasible_affordable` is reported separately.

252 answer generations plus A messages, plus a planning pass on extra-compute. Resume is on for the **corrected** folder only. Download `/kaggle/working/results_stage_plan0_corrected` during the run.

Do not drop individual failures. These are 12 families with three versions each, not 36 independent scenarios.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan0.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan0.py").is_file() and (cand / "data" / "plan_dev.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan0.py and data/plan_dev.jsonl. Re-upload the repo zip.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_plan0.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("plan_dev", sum(1 for _ in (REPO / "data" / "plan_dev.jsonl").open(encoding="utf-8")))
assert not (REPO / "data" / "plan_test.jsonl").exists()

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan0_corrected")
print("Do not resume results_stage_plan0")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan0", "--split", "plan_dev"])
print("plan_dev exit", rc)

In [ ]:
src = WORK / "results_stage_plan0_corrected"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan0_corrected" / "run"
print("reading", src)
for name in ("summary.json", "manifest.json"):
    path = src / name
    if path.is_file():
        print("====", name)
        print(path.read_text(encoding="utf-8")[:8000])